# Long-Term Memory Prioritization Demo (CosmosDB)

This notebook is the notebook-converted version of `demo/09_itemized_insights_cosmos.py`.

It demonstrates a memory system where insights compete for limited long-term storage capacity using:
- **Recency**: newer insights receive temporary priority
- **Frequency**: cited/reused insights get stronger over time
- **Forgetting**: old unused insights decay
- **Bounded memory**: only top-N insights are retained

## Environment prerequisites
- `AZURE_OPENAI_ENDPOINT`
- `AZURE_OPENAI_API_KEY`
- `COSMOS_ENDPOINT` and `COSMOS_KEY` (or Azure AD if key is omitted)

In [ ]:
import asyncio
import os
import sys
from pathlib import Path
from datetime import datetime
from typing import Dict, List

from dotenv import load_dotenv

# Resolve project root whether launched from workspace root or demo folder.
cwd = Path.cwd().resolve()
BASE_DIR = cwd.parent if cwd.name == 'demo' else cwd
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

load_dotenv(BASE_DIR / '.env', override=True)

from openai import AzureOpenAI
from memory.db.cosmos_backend import CosmosDBDatabase
from memory.db.base import ContainerType
from memory.providers.embedding import OpenAIEmbeddingProvider
from memory.core.reflection import Reflection, ReflectionConfig
from memory.core.insight_items import (
    LongTermInsightItem,
    InsightIdGenerator,
    rank_insights,
    calculate_retention_score,
    SessionAnalysisWithCitations,
    build_context_with_ids,
)

USER_ID = 'memory_priority_demo_cosmos'
MAX_INSIGHTS = 5

client = AzureOpenAI(
    azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
    api_key=os.environ.get('AZURE_OPENAI_API_KEY'),
    api_version='2024-10-21',
)

TIMELINE = [
    {
        'session_id': 'month-1',
        'simulated_date': datetime(2025, 1, 15),
        'title': 'Month 1: Initial Consultation',
        'summary': 'Alex is 28, software engineer earning $120k. Wants to start investing. Very risk-averse due to parents 2008 losses. Prefers bonds and savings. Has $10,000 emergency fund. Interested in Roth IRA.',
        'turns': [
            ('user', "I'm Alex, 28, making $120k. I want to start investing but I'm scared of losing money."),
            ('assistant', "That's understandable. Let's start with your risk tolerance and goals."),
            ('user', "My parents lost a lot in 2008. I want safe investments - bonds, savings accounts."),
            ('assistant', "With your fear of losses, we can start conservatively. You mentioned a Roth IRA?"),
            ('user', "Yes, I want to learn about Roth IRAs. I have $10k saved for emergencies."),
        ],
    },
    {
        'session_id': 'month-2',
        'simulated_date': datetime(2025, 2, 20),
        'title': 'Month 2: Roth IRA Setup',
        'summary': 'Alex opened a Roth IRA last month and asks about contribution limits. Still conservative with money market allocation. Mentions saving for sisters wedding.',
        'turns': [
            ('user', "I opened the Roth IRA! How much can I contribute this year?"),
            ('assistant', "Great! The 2025 limit is $7,000. Given your conservative preference, what did you choose?"),
            ('user', "I put it in a money market fund. I know it's low return but I can't handle volatility."),
            ('assistant', "That's fine to start. Any other financial goals coming up?"),
            ('user', "My sister's wedding is in April. I need to save about $2,000 for travel and gift."),
        ],
    },
    {
        'session_id': 'month-3',
        'simulated_date': datetime(2025, 3, 25),
        'title': 'Month 3: Tax Season Questions',
        'summary': 'Alex asks tax implications of Roth contributions and plans to max out the Roth IRA. Risk profile still conservative.',
        'turns': [
            ('user', "Quick tax question - do my Roth IRA contributions reduce my taxable income?"),
            ('assistant', "No, Roth contributions are post-tax. But withdrawals in retirement are tax-free."),
            ('user', "I think I want to max out my Roth this year."),
            ('assistant', "Good plan. Still comfortable with money market holdings?"),
            ('user', "Yes, not ready for stocks yet. Maybe next year."),
        ],
    },
    {
        'session_id': 'month-4',
        'simulated_date': datetime(2025, 5, 10),
        'title': 'Month 4: Post-Wedding, Promotion News',
        'summary': 'Wedding complete. Alex promoted to $150k and starts reconsidering risk tolerance. Asks about target-date funds.',
        'turns': [
            ('user', "Big news! Wedding was great, and I just got promoted to $150k!"),
            ('assistant', "Congratulations! How are you feeling about finances now?"),
            ('user', "More confident. Maybe I can handle some risk."),
            ('assistant', "Would you like to explore middle-ground options?"),
            ('user', "Yes, target-date funds seem balanced."),
        ],
    },
    {
        'session_id': 'month-5',
        'simulated_date': datetime(2025, 6, 15),
        'title': 'Month 5: Risk Tolerance Shift',
        'summary': 'Alex moves Roth IRA to target-date 2060 fund and starts taxable account planning for home down payment.',
        'turns': [
            ('user', "I moved my Roth to a 2060 target-date fund."),
            ('assistant', "Great step. How do you feel about it?"),
            ('user', "Good. Now I want a taxable account too."),
            ('assistant', "Any goal for the taxable account?"),
            ('user', "House down payment in 3-4 years."),
        ],
    },
    {
        'session_id': 'month-6',
        'simulated_date': datetime(2025, 7, 20),
        'title': 'Month 6: Current State & Planning',
        'summary': 'Alex keeps house fund conservative while becoming aggressive for long-term retirement and considers higher 401k contributions.',
        'turns': [
            ('user', "I started the house fund with $5,000 and kept it conservative."),
            ('assistant', "Good time-horizon matching. How is your 401k?"),
            ('user', "Should I increase 401k contributions now?"),
            ('assistant', "At $150k, maxing 401k is ideal if cash flow allows."),
            ('user', "I'll do that."),
        ],
    },
]

print(f'Base directory: {BASE_DIR}')
print('Setup complete.')

In [ ]:
def print_section(title: str, char: str = '=') -> None:
    width = 70
    print(f'\n{char * width}')
    print(f' {title}')
    print(f'{char * width}')


def print_insight_table(items: List[LongTermInsightItem], now: datetime, title: str = 'Current Insights') -> None:
    print(f'\n{title}:')
    print('-' * 95)
    print(f"{'ID':<10} {'Score':>6} {'Access':>7} {'Age':>8} {'Importance':>10}  {'Text':<40}")
    print('-' * 95)

    ranked = rank_insights(items, now)
    for item, score in ranked:
        age_days = (now - item.date_added).days
        age_str = f'{age_days}d' if age_days < 30 else f'{age_days // 30}m {age_days % 30}d'
        text = item.insight_text[:38] + '..' if len(item.insight_text) > 38 else item.insight_text
        print(f"{item.id:<10} {score:>6.2f} {item.access_count:>7} {age_str:>8} {item.importance:>10}  {text}")
    print('-' * 95)


async def get_insight_items(db: CosmosDBDatabase, user_id: str) -> List[LongTermInsightItem]:
    items = await db.query(
        container=ContainerType.INSIGHTS,
        filters={'user_id': user_id, 'insight_type': 'long_term_item'},
        order_by='-created_at',
    )

    result = []
    for item_data in items:
        try:
            result.append(LongTermInsightItem.from_dict(item_data))
        except Exception as exc:
            print(f'  Warning: Could not parse insight item: {exc}')
    return result


async def cleanup_demo_data(db: CosmosDBDatabase, user_id: str) -> None:
    items = await db.query(container=ContainerType.INSIGHTS, filters={'user_id': user_id})
    for item in items:
        await db.delete(container=ContainerType.INSIGHTS, document_id=item['id'], partition_key=user_id)
    print(f'[Cleaned up {len(items)} existing insights for demo user]')


async def prune_insights(
    db: CosmosDBDatabase,
    user_id: str,
    items: List[LongTermInsightItem],
    max_items: int,
    now: datetime,
) -> tuple[list[LongTermInsightItem], list[LongTermInsightItem]]:
    if len(items) <= max_items:
        return items, []

    ranked = rank_insights(items, now)
    kept = [item for item, _ in ranked[:max_items]]
    pruned = [item for item, _ in ranked[max_items:]]

    for item in pruned:
        await db.delete(container=ContainerType.INSIGHTS, document_id=item.id, partition_key=user_id)

    return kept, pruned


async def run_session_with_simulated_time(
    reflection: Reflection,
    db: CosmosDBDatabase,
    emb_provider: OpenAIEmbeddingProvider,
    user_id: str,
    session: Dict,
    simulated_now: datetime,
) -> Dict:
    from memory.prompts import SESSION_ANALYSIS_WITH_CITATIONS_PROMPT

    existing_items = await get_insight_items(db, user_id)
    existing_context = build_context_with_ids(existing_items)

    turns_text = '\n'.join([f"{role}: {content}" for role, content in session['turns']])
    full_context = f"{session['summary']}\n\nConversation:\n{turns_text}"

    prompt = SESSION_ANALYSIS_WITH_CITATIONS_PROMPT.format(
        existing_insights_context=existing_context,
        session_content=full_context,
    )

    try:
        analysis = reflection._call_llm_with_json(
            system_prompt='You are an expert session analysis assistant with memory tracking.',
            user_prompt=prompt,
            output_model=SessionAnalysisWithCitations,
        )
    except Exception as exc:
        print(f'  Error: {exc}')
        return {
            'session_summary': 'Error',
            'key_topics': [],
            'new_insights': [],
            'cited_insight_ids': [],
            'has_meaningful_content': False,
        }

    cited_ids = []
    for citation in analysis.cited_insights:
        cited_ids.append(citation.insight_id)
        doc = await db.get_by_id(
            container=ContainerType.INSIGHTS,
            document_id=citation.insight_id,
            partition_key=user_id,
        )
        if doc and doc.get('insight_type') == 'long_term_item':
            doc['access_count'] = doc.get('access_count', 0) + 1
            doc['last_accessed'] = simulated_now.isoformat()
            await db.upsert(container=ContainerType.INSIGHTS, document=doc, partition_key=user_id)

    id_gen = InsightIdGenerator([item.id for item in existing_items])
    new_insight_items = []

    for insight in analysis.new_insights:
        item = LongTermInsightItem(
            id=id_gen.next_id(),
            user_id=user_id,
            agent_id='default',
            insight_text=insight.insight_text,
            category=insight.category,
            confidence=insight.confidence,
            importance=insight.importance,
            date_added=simulated_now,
            last_accessed=simulated_now,
            access_count=0,
            source_session_ids=[session['session_id']],
        )
        new_insight_items.append(item)

    for item in new_insight_items:
        item.embedding = emb_provider.get_embedding(item.insight_text)
        await db.upsert(container=ContainerType.INSIGHTS, document=item.to_dict(), partition_key=user_id)

    return {
        'session_summary': analysis.session_summary,
        'key_topics': analysis.key_topics,
        'new_insights': [i.to_dict() for i in new_insight_items],
        'cited_insight_ids': cited_ids,
        'has_meaningful_content': analysis.has_meaningful_content,
    }


print('Helper functions are ready.')

## Cell 4: What Happens During Execution

Cell 5 executes a full six-session simulation over synthetic monthly timestamps, while persisting everything in CosmosDB.

### Step-by-step flow
1. **Connect and initialize CosmosDB**
- Uses `COSMOS_ENDPOINT` and `COSMOS_KEY` when available.
- If key is missing, it attempts Azure AD (`DefaultAzureCredential`).

2. **Reset demo user state**
- Deletes prior long-term insights for `memory_priority_demo_cosmos` so each run starts clean.

3. **Run each timeline session**
- Builds prompt context from existing insights with stable IDs.
- Calls reflection analysis to extract **new insights** and **citations to old insights**.
- Citations increment `access_count` and refresh `last_accessed`.

4. **Apply bounded-memory pruning**
- If insight count exceeds `MAX_INSIGHTS=5`, rank by retention score and delete weaker items.
- Demonstrates forgetting behavior under constrained capacity.

5. **Print final evolution summary**
- Shows month-by-month top insights, score dynamics, and access frequency.

6. **Cleanup**
- Deletes demo insights again to avoid polluting persistent data.

### How to read the score table
- **Score**: combined retention priority (higher means more likely to survive).
- **Access**: number of citation hits from later sessions.
- **Age**: days/months since the insight was first added.
- **Importance**: model-assigned salience from extracted insight metadata.

In [ ]:
async def run_demo() -> None:
    print_section('LONG-TERM MEMORY PRIORITIZATION DEMO (CosmosDB)')
    print(
        f"\nThis demo simulates 6 months of client sessions using Azure CosmosDB.\n\n"
        f"Key concepts demonstrated:\n"
        f"- RECENCY: New insights start with a grace-period boost\n"
        f"- FREQUENCY: Cited insights gain strength (access_count rises)\n"
        f"- FORGETTING: Old, uncited insights decay over time\n"
        f"- BOUNDED MEMORY: Only {MAX_INSIGHTS} insights are retained\n"
    )

    cosmos_endpoint = os.environ.get('COSMOS_ENDPOINT') or os.environ.get('AZURE_COSMOS_ENDPOINT')
    cosmos_key = os.environ.get('COSMOS_KEY') or os.environ.get('AZURE_COSMOS_KEY')

    if not cosmos_endpoint:
        raise ValueError('COSMOS_ENDPOINT/AZURE_COSMOS_ENDPOINT is required.')

    print('[Connecting to Azure CosmosDB...]')
    if cosmos_key:
        print('  Using key-based authentication')
        db = CosmosDBDatabase(
            endpoint=cosmos_endpoint,
            key=cosmos_key,
            database_name=os.environ.get('COSMOS_DATABASE', os.environ.get('AZURE_COSMOS_DATABASE_NAME', 'agent_memory_db')),
        )
    else:
        print('  No key found, trying Azure AD authentication...')
        from azure.identity import DefaultAzureCredential

        db = CosmosDBDatabase(
            endpoint=cosmos_endpoint,
            credential=DefaultAzureCredential(),
            database_name=os.environ.get('COSMOS_DATABASE', os.environ.get('AZURE_COSMOS_DATABASE_NAME', 'agent_memory_db')),
        )

    await db.initialize()
    print('[Connected successfully]')

    await cleanup_demo_data(db, USER_ID)

    emb_provider = OpenAIEmbeddingProvider(
        openai_client=client,
        model=os.environ.get('AZURE_OPENAI_EMB_DEPLOYMENT', 'text-embedding-ada-002'),
    )

    reflection = Reflection(
        agent_id='default',
        database=db,
        embedding_provider=emb_provider,
        chat_client=client,
        config=ReflectionConfig(
            PROCESSING_MODEL=(
                os.environ.get('AZURE_OPENAI_PROCESSING_MODEL')
                or os.environ.get('AZURE_OPENAI_REASONING_MODEL')
                or 'gpt-4o-mini'
            )
        ),
    )

    insight_history = []

    try:
        for month_idx, session in enumerate(TIMELINE, 1):
            simulated_now = session['simulated_date']
            print_section(f"SESSION {month_idx}: {session['title']}", '=')
            print(f"Simulated Date: {simulated_now.strftime('%B %d, %Y')}")

            items_before = await get_insight_items(db, USER_ID)
            if items_before:
                print_insight_table(items_before, simulated_now, 'Memory State BEFORE Session')
            else:
                print('\n[No existing insights - this is the first session]')

            print('\n[Processing session...]')
            result = await run_session_with_simulated_time(
                reflection, db, emb_provider, USER_ID, session, simulated_now
            )

            print(f"\nSummary: {result['session_summary'][:100]}...")
            print(f"New insights: {len(result['new_insights'])}")
            for ins in result['new_insights']:
                print(f"  - [{ins['id']}] {ins['insight_text'][:50]}...")

            if result['cited_insight_ids']:
                print(f"Cited existing: {result['cited_insight_ids']}")

            items_after = await get_insight_items(db, USER_ID)
            if len(items_after) > MAX_INSIGHTS:
                print(f"\nCapacity exceeded ({len(items_after)} > {MAX_INSIGHTS}). Pruning...")
                kept, pruned = await prune_insights(db, USER_ID, items_after, MAX_INSIGHTS, simulated_now)

                print('Forgotten:')
                for item in pruned:
                    score = calculate_retention_score(item, simulated_now)
                    age = (simulated_now - item.date_added).days
                    print(f"  - [{item.id}] score={score:.2f} age={age}d access={item.access_count}: {item.insight_text[:35]}...")

                print('Retained:')
                for item in kept:
                    score = calculate_retention_score(item, simulated_now)
                    age = (simulated_now - item.date_added).days
                    print(f"  - [{item.id}] score={score:.2f} age={age}d access={item.access_count}: {item.insight_text[:35]}...")

                items_after = kept

            print_insight_table(items_after, simulated_now, 'Memory State AFTER Session (Top 5)')

            insight_history.append({
                'month': month_idx,
                'date': simulated_now,
                'insights': [
                    (
                        i.id,
                        i.insight_text[:25] + '...',
                        i.access_count,
                        calculate_retention_score(i, simulated_now),
                    )
                    for i in sorted(
                        items_after,
                        key=lambda x: calculate_retention_score(x, simulated_now),
                        reverse=True,
                    )
                ],
            })

        print_section('DEMO COMPLETE - MEMORY EVOLUTION SUMMARY', '=')
        print('\nHow insights evolved over 6 months:')
        print('-' * 80)
        for record in insight_history:
            print(f"\nMonth {record['month']} ({record['date'].strftime('%B %Y')}):")
            for insight_id, text, access, score in record['insights']:
                print(f"  [{insight_id}] score={score:.2f} access={access}: {text}")

        print('\n' + '=' * 70)
        print('KEY OBSERVATIONS (CosmosDB):')
        print('=' * 70)
        print(
            'Same memory prioritization behavior as SQLite\n'
            'Scales with CosmosDB distribution and reliability\n'
            'Vector search supports semantic recall\n'
            'Bounded memory keeps the most relevant insights'
        )

    finally:
        await cleanup_demo_data(db, USER_ID)
        await db.close()
        print('\n[Demo data cleaned up]')


await run_demo()